# Strands Agents with AgentCore Memory (Long-term) using tools

# Strands Agents 与 AgentCore Memory（长期）使用工具

## Overview

## 概述

This notebook demonstrates how to implement long-term memory capabilities for conversational AI agents using Strands and AgentCore Memory. You'll learn how to extract and consolidate important information from short-term interactions, enabling an agent to recall key details across multiple conversation sessions over time.

本笔记本演示如何使用 Strands 和 AgentCore Memory 为对话式 AI 代理实现长期记忆功能。您将学习如何从短期交互中提取和整合重要信息，使代理能够在多个对话会话中随时间记住关键细节。

## Tutorial Details

## 教程详情

**Use Case:** Culinary Assistant with Persistent Memory

**用例：** 具有持久记忆的烹饪助手

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long term Conversational                                                         |
| Agent type          | Culinary Assistant                                                               |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore 'User Preferences' Memory Extraction, Memory Tool for storing and retrieving Memory              |
| Example complexity  | Beginner                                                                     |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 长期对话式                                                                        |
| 代理类型             | 烹饪助手                                                                          |
| 代理框架             | Strands Agents                                                                   |
| LLM 模型            | Anthropic Claude Haiku 4.5                                                       |
| 教程组件             | AgentCore"用户偏好"记忆提取、用于存储和检索记忆的记忆工具                            |
| 示例复杂度           | 初级                                                                              |

You'll learn to:

您将学习：

- Configure AgentCore Memory with extraction strategies for long-term retention
- Hydrate memory with previous conversation history
- Use long-term memory to deliver personalized experiences across conversation sessions
- Integrate Strands Agent Framework with the AgentCore Memory tool

- 配置具有长期保留提取策略的 AgentCore Memory
- 使用之前的对话历史填充记忆
- 使用长期记忆跨对话会话提供个性化体验
- 将 Strands Agent 框架与 AgentCore Memory 工具集成

## Scenario Context

## 场景背景

In this tutorial, you'll step into the role of a Culinary Assistant designed to deliver highly personalized restaurant recommendations. By leveraging AgentCore Memory's long-term retention and automatic information extraction, the agent can remember user preferences—such as dietary choices and favorite cuisines—across multiple conversations. This persistent memory enables the agent to provide tailored suggestions and a seamless user experience, even as conversations span days or weeks. The scenario demonstrates how structured memory organization and configurable strategies empower conversational AI to move beyond short-term recall, creating truly engaging and context-aware interactions.

在本教程中，您将扮演一个烹饪助手的角色，旨在提供高度个性化的餐厅推荐。通过利用 AgentCore Memory 的长期保留和自动信息提取，代理可以跨多个对话记住用户偏好（如饮食选择和喜爱的菜系）。这种持久记忆使代理能够提供定制建议和无缝的用户体验，即使对话跨越数天或数周。该场景演示了结构化记忆组织和可配置策略如何使对话式 AI 超越短期回忆，创建真正引人入胜和上下文感知的交互。


## Architecture

## 架构

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Prerequisites

## 前提条件

To execute this tutorial you will need:

执行本教程您需要：

- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK

- Python 3.10+
- 具有 Amazon Bedrock AgentCore Memory 权限的 AWS 凭证
- Amazon Bedrock AgentCore SDK

Let's get started by setting up our environment and creating our long-term memory resource with the appropriate extraction strategy!

让我们开始设置环境并使用适当的提取策略创建长期记忆资源！

## Step 1: Environment set up

## 第一步：环境设置

Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

让我们开始导入所有必要的库并定义客户端，使本笔记本正常工作。

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import time
import logging
import time
from datetime import datetime

Define the region and the role with the appropiate permissions for Amazon Bedrock models and AgentCore

定义区域和具有 Amazon Bedrock 模型和 AgentCore 适当权限的角色

In [ ]:
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("culinary-memory")

region = os.getenv('AWS_REGION', 'us-west-2')

## Step 2: Creating Memory with Long-Term Strategies

## 第二步：创建具有长期策略的记忆

In this section, we'll create a memory resource configured with long-term memory capabilities. Unlike our previous short-term memory example, this implementation includes specific memory strategies that enable consolidated information retention.

在本节中，我们将创建一个配置了长期记忆功能的记忆资源。与之前的短期记忆示例不同，此实现包含特定的记忆策略，可实现整合信息保留。

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)

memory_name = "CulinaryAssistant"
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Long-Term Memory...")

    # We use a more descriptive name for our long-term memory resource
    memory_name = memory_name

    # Create memory with user preference strategy
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Culinary Assistant Agent with long term memory",
        strategies=[{
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "UserPreferences",
                        "description": "Captures user preferences",
                        "namespaces": ["user/{actorId}/preferences"]
                    }
                }],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )

    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Understanding Long-Term Memory Strategies

### 理解长期记忆策略

The key difference in this memory creation is the addition of a **memory strategy**. Let's break down the components:

此记忆创建的关键区别是添加了**记忆策略**。让我们分解这些组件：

#### 1. User Preference Memory Strategy

#### 1. 用户偏好记忆策略

This strategy automatically identifies and extracts user preferences from conversations:

此策略自动从对话中识别和提取用户偏好：

```python
"userPreferenceMemoryStrategy": {
    "name": "UserPreferences",
    "description": "Captures user preferences",
    "namespaces": ["user/{actorId}/preferences"]
}
```

#### 2. Memory Namespaces

#### 2. 记忆命名空间

The `namespaces` parameter defines where extracted information is stored:

`namespaces` 参数定义提取的信息存储位置：

```python
"namespaces": ["user/{actorId}/preferences"]
```

This memory strategy creates a more sophisticated memory system that doesn't just remember conversations, but actually understands and organizes the important information within those conversations for future use.

此记忆策略创建了一个更复杂的记忆系统，它不仅记住对话，还实际理解和组织对话中的重要信息以供将来使用。

## Step 3: Saving Previous Conversations to Memory

## 第三步：将之前的对话保存到记忆

In this section, we'll demonstrate how to hydrate the short-term memory, which automatically triggers the long-term memory extraction process behind the scenes.

在本节中，我们将演示如何填充短期记忆，这会自动在后台触发长期记忆提取过程。

### Hydrating Short-Term Memory

### 填充短期记忆

When we save conversations to a memory resource configured with extraction strategies, the system automatically processes this information for long-term retention without requiring additional code.

当我们将对话保存到配置了提取策略的记忆资源时，系统会自动处理此信息以进行长期保留，无需额外代码。

In [ ]:
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"foodie-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"user/{actor_id}/preferences"

In [ ]:
previous_messages = [
    ("Hi, I'm John", "USER"),
    ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
    ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
    ("That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?", "ASSISTANT"),
    ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
    ("Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?", "ASSISTANT"),
    ("I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.", "USER"),
    ("Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!", "ASSISTANT"),
    ("I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes", "USER"),
    ("I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?", "ASSISTANT"),
    ("Hey, I appreciate food assistants with good taste", "USER"),
    ("Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?", "ASSISTANT")
]

In [ ]:
print("\nHydrating short term memory with previous conversations...")

# Save the conversation history to short-term memory
initial = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print("✓ Conversation saved in short term memory")

Let's make sure the event containing the conversation messages was stored correctly.

让我们确保包含对话消息的事件已正确存储。

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=5
)
events

This cell configures the logging system to display informative messages during execution, helping us track what's happening as our code runs.

此单元格配置日志系统以在执行期间显示信息性消息，帮助我们跟踪代码运行时发生的情况。

### What Happens Behind the Scenes

### 幕后发生了什么

After the `create_event` call, the following occurs automatically:

在 `create_event` 调用之后，以下操作会自动发生：

1. **Short-Term Storage**: The complete conversation is saved in raw form
2. **Extraction Trigger**: The memory system detects that this memory has the UserPreference strategy configured
3. **Background Processing**: Without any additional code, the system:
   - Analyzes the conversation for preference indicators
   - Identifies statements like "I'm vegetarian" and "I really enjoy Italian cuisine"
   - Extracts these preferences into structured data
4. **Long-Term Consolidation**: The extracted preferences are saved in the configured namespace (`user/{actorId}/preferences`)

1. **短期存储**：完整对话以原始形式保存
2. **提取触发**：记忆系统检测到此记忆配置了 UserPreference 策略
3. **后台处理**：无需任何额外代码，系统：
   - 分析对话中的偏好指标
   - 识别诸如"我是素食者"和"我非常喜欢意大利菜"等陈述
   - 将这些偏好提取为结构化数据
4. **长期整合**：提取的偏好保存在配置的命名空间中（`user/{actorId}/preferences`）

Extraction and consolidation happen automatically - we only need to mantain a conversation with the agent or hydrate the short-term memory, and the strategies we configured during memory creation take care of the rest.

提取和整合自动发生 - 我们只需要与代理保持对话或填充短期记忆，我们在记忆创建期间配置的策略会处理其余部分。

This automatic process ensures that important information is preserved in long-term memory even after the short-term conversation records expire.

此自动过程确保即使在短期对话记录过期后，重要信息也会保存在长期记忆中。

## Retrieving Long-Term Memories

## 检索长期记忆

In this section, we'll explore how to access the extracted preferences that have been stored in long-term memory. Unlike short-term memory retrieval which focuses on conversation turns, long-term memory retrieval focuses on accessing structured information that has been extracted and consolidated.

在本节中，我们将探讨如何访问已存储在长期记忆中的提取偏好。与专注于对话轮次的短期记忆检索不同，长期记忆检索专注于访问已提取和整合的结构化信息。

### Accessing User Preferences from Long-Term Memory

### 从长期记忆中访问用户偏好

To retrieve information from long-term memory, we use the namespace structure defined during memory creation:

要从长期记忆中检索信息，我们使用在记忆创建期间定义的命名空间结构：

In [ ]:
# Adding a 30s wait to ensure the memory extraction has time to process the event
time.sleep(30)

try:
    # Query the memory system for food preferences
    food_preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="food preferences",
        top_k=3  # Return up to 3 most relevant results
    )

    if food_preferences:
        print(f"Retrieved {len(food_preferences)} relevant preference records:")
        for i, record in enumerate(food_preferences):
            print(f"\nMemory {i+1}:")
            print(f"- Content: {record.get('content', 'Not specified')}")
    else:
        print("No matching preference records found.")

except Exception as e:
    print(f"Error retrieving preference records: {e}")

This method enables the retrieval of relevant memories when needed. Now we learned the basics let's build up our agent!

此方法可在需要时检索相关记忆。现在我们学习了基础知识，让我们构建代理！

## Step 4: Creating the agent 

## 第四步：创建代理

In this section, we'll explore how to integrate AgentCore Memory with a Strands Agent using the native `agent_core_memory` tool.

在本节中，我们将探讨如何使用原生 `agent_core_memory` 工具将 AgentCore Memory 与 Strands Agent 集成。

#### Setting Up the Agent with Long term Memory Capabilities

#### 设置具有长期记忆功能的代理

To create a memory-enabled agent, we'll use the Strands framework and connect it to our AgentCore Memory resource

要创建支持记忆的代理，我们将使用 Strands 框架并将其连接到我们的 AgentCore Memory 资源

In [ ]:
from strands import tool, Agent
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

In [ ]:
system_prompt = f"""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations

"""

In [ ]:
provider = AgentCoreMemoryToolProvider(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    namespace=namespace
)

agent = Agent(tools=provider.tools, model="global.anthropic.claude-haiku-4-5-20251001-v1:0",system_prompt=system_prompt)

As we have already populated our short term and long term memory, let's directly retrieve the memory from the agent!

由于我们已经填充了短期和长期记忆，让我们直接从代理检索记忆！

In [ ]:
agent("Give me restaurant recommendations in Irvine based on my food preferences")

The agent should have used the retrieve_memory_records method to retrieve the user's memories.

代理应该已使用 retrieve_memory_records 方法检索用户的记忆。

Great! You know have a working Strands Agent capable of retrieving memories from the AgentCore Long Term Memory!

太好了！您现在有了一个能够从 AgentCore 长期记忆中检索记忆的工作 Strands Agent！

## Clean up

## 清理

Let's delete the memory to clean up the resources used in this notebook.

让我们删除记忆以清理本笔记本中使用的资源。

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)